# RSI(2) Rebound on SPY
## Strategy Brief
The RSI(2) Rebound strategy is a mean reversion trading strategy applied on SPY (S&P 500 ETF). It uses the 2-period Relative Strength Index (RSI) to identify short-term oversold conditions, predicting a price rebound. The strategy buys SPY when the RSI(2) falls below a certain threshold, indicating oversold conditions, and sells when it rises above another threshold, indicating a rebound. Historical backtesting suggests that this strategy can outperform a simple buy-and-hold approach by capitalizing on short-term price fluctuations.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
Define the trading parameters and constants for the RSI(2) Rebound strategy.

In [ ]:
RSI_PERIOD = 2
RSI_OVERSOLD = 10
RSI_OVERBOUGHT = 90
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
TICKER = 'SPY'

### PHASE 2 - Data Exploration
Download historical SPY data, compute the RSI(2) indicator, and visualize it overlaid on the price chart.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download(TICKER, start=START_DATE, end=END_DATE)

# Compute RSI(2)
delta = data['Close'].diff()
gain = (delta.where(delta > 0, 0)).rolling(RSI_PERIOD).mean()
loss = (-delta.where(delta < 0, 0)).rolling(RSI_PERIOD).mean()
rs = gain / loss
rsi = 100 - (100 / (1 + rs))
data['RSI'] = rsi

# Plot Closing Prices and RSI
plt.figure(figsize=(14, 7))
plt.subplot(2, 1, 1)
plt.plot(data['Close'], label='Close')
plt.title('SPY Closing Prices')
plt.legend()
plt.subplot(2, 1, 2)
plt.plot(data['RSI'], label='RSI(2)', color='orange')
plt.axhline(y=RSI_OVERSOLD, color='red', linestyle='--')
plt.axhline(y=RSI_OVERBOUGHT, color='green', linestyle='--')
plt.title('RSI(2)')
plt.legend()
plt.tight_layout()
plt.show()

### PHASE 3 - Strategy Engineering
Create the trading signals, entry/exit logic based on RSI(2), and determine the positions.

In [ ]:
# Generate signals
data['Signal'] = 0
# Buy signal
data.loc[data['RSI'] < RSI_OVERSOLD, 'Signal'] = 1
# Sell signal
data.loc[data['RSI'] > RSI_OVERBOUGHT, 'Signal'] = -1

# Determine positions
data['Position'] = data['Signal'].shift(1).fillna(0)

### PHASE 4 - Coding & Backtesting
Calculate daily returns, apply the strategy, and plot the equity curve.

In [ ]:
# Calculate daily returns
data['Market_Return'] = data['Close'].pct_change()
data['Strategy_Return'] = data['Market_Return'] * data['Position']

data['Equity_Curve'] = (1 + data['Strategy_Return']).cumprod()
data['Market_Equity'] = (1 + data['Market_Return']).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(data['Equity_Curve'], label='Strategy Equity Curve')
plt.plot(data['Market_Equity'], label='Market Equity Curve', linestyle='--')
plt.title('Equity Curve')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
Evaluate the performance of the strategy using key metrics and compare it to a buy-and-hold approach.

In [ ]:
def calculate_cagr(equity_curve):
    n = len(equity_curve) / 252  # Assuming 252 trading days in a year
    return (equity_curve[-1] / equity_curve[0]) ** (1/n) - 1

def calculate_sharpe(strategy_returns):
    return strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)

def calculate_sortino(strategy_returns):
    downside_returns = strategy_returns[strategy_returns < 0]
    return strategy_returns.mean() / downside_returns.std() * np.sqrt(252)

def calculate_max_drawdown(equity_curve):
    roll_max = equity_curve.cummax()
    drawdown = (equity_curve - roll_max) / roll_max
    return drawdown.min()

cagr_strategy = calculate_cagr(data['Equity_Curve'])
cagr_market = calculate_cagr(data['Market_Equity'])
sharpe_strategy = calculate_sharpe(data['Strategy_Return'].dropna())
sharpe_market = calculate_sharpe(data['Market_Return'].dropna())
sortino_strategy = calculate_sortino(data['Strategy_Return'].dropna())
max_drawdown_strategy = calculate_max_drawdown(data['Equity_Curve'])
max_drawdown_market = calculate_max_drawdown(data['Market_Equity'])

performance = pd.DataFrame({
    'Strategy': [cagr_strategy, sharpe_strategy, sortino_strategy, max_drawdown_strategy],
    'Market': [cagr_market, sharpe_market, 'N/A', max_drawdown_market]
}, index=['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Max Drawdown'])

performance

### PHASE 6 - Deploy & Monitor
Create a function to download the latest data, compute the RSI(2) signal, and determine today's position.

In [ ]:
def get_latest_signal(ticker, period=RSI_PERIOD, oversold=RSI_OVERSOLD, overbought=RSI_OVERBOUGHT):
    # Download last 60 days of data
    recent_data = yf.download(ticker, period='60d')
    
    # Compute RSI
    delta = recent_data['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    
    # Determine signal
    if rsi.iloc[-1] < oversold:
        position = 'Buy'
    elif rsi.iloc[-1] > overbought:
        position = 'Sell'
    else:
        position = 'Hold'
    
    print(f"Today's RSI(2) for {ticker}: {rsi.iloc[-1]:.2f}, Position: {position}")

get_latest_signal(TICKER)